In [1]:
# Cell 0: Install dependencies
import subprocess
subprocess.run(['pip', 'install', 'torch', 'torchvision', 'torchaudio'], check=True)
subprocess.run(['pip', 'install', 'transformers', 'accelerate>=1.1.0'], check=True)
subprocess.run(['pip', 'install', 'scikit-learn', 'plotly', 'pandas', 'numpy'], check=True)
print("✅ Semua library terinstall! Restart kernel lalu jalankan Cell 1.")

✅ Semua library terinstall! Restart kernel lalu jalankan Cell 1.


In [1]:
# Cell 1: Import Libraries & Setup
import pandas as pd
import numpy as np
import torch
import os, gc, json, pickle, warnings, re
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.utils import shuffle
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
import plotly.express as px
import plotly.graph_objects as go

os.environ["WANDB_DISABLED"] = "true"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings('ignore')

for folder in ['flask_components/models', 'flask_components/data',
               'flask_components/visualizations', 'flask_components/results']:
    os.makedirs(folder, exist_ok=True)

print(f"✅ Setup selesai | PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}")
print("📁 Folder flask_components/ siap")

✅ Setup selesai | PyTorch: 2.7.1+cu118 | CUDA: False
📁 Folder flask_components/ siap


In [2]:
# Cell 2: Load Dataset
# Pastikan clean_news_dataset.csv ada di folder yang sama dengan notebook
df = pd.read_csv('clean_news_dataset.csv')

print(f"📊 Shape: {df.shape} | Kolom: {list(df.columns)}")
print(f"\nMissing values:\n{df.isnull().sum()}")
display(df)

# Save untuk Flask
dataset_info = {
    'total_rows': int(len(df)),
    'columns': list(df.columns),
    'missing_values': {k: int(v) for k, v in df.isnull().sum().to_dict().items()},
    'label_distribution': {k: int(v) for k, v in df['label'].value_counts().to_dict().items()},
    'source_distribution': {k: int(v) for k, v in df['sumber'].value_counts().to_dict().items()},
    'title_length_stats': {
        'mean': float(df['title_length'].mean()), 'std': float(df['title_length'].std()),
        'min': int(df['title_length'].min()), 'max': int(df['title_length'].max())
    }
}
with open('flask_components/data/dataset_info.json', 'w', encoding='utf-8') as f:
    json.dump(dataset_info, f, indent=2, ensure_ascii=False)
df.to_csv('flask_components/data/raw_dataset.csv', index=False)
print("💾 dataset_info.json & raw_dataset.csv disimpan")

📊 Shape: (248, 5) | Kolom: ['judul', 'label', 'sumber', 'kategori', 'title_length']

Missing values:
judul           0
label           0
sumber          0
kategori        0
title_length    0
dtype: int64


,judul,label,sumber,kategori,title_length
0,SALAH Video Pengakuan Kekalahan Israel,FAKE,TurnBackHoax.id,News,38
1,SALAH Video Warga Israel Mengungsi di Gurun Pasir,FAKE,TurnBackHoax.id,News,49
2,SALAH Artikel CNBC Indonesia: Raja Salman Sebu...,FAKE,TurnBackHoax.id,News,87
3,PENIPUAN Tautan Pendaftaran CPNS Kemenhub,FAKE,TurnBackHoax.id,News,41
4,SALAH Video Monster Drone Buatan Iran,FAKE,TurnBackHoax.id,News,37
...,...,...,...,...,...
243,Airlangga dorong pengembangan investasi Temase...,REAL,Antara News,News,69
244,Vale (INCO) jadwalkan ulang RUPSLB penunjukan ...,REAL,Antara News,News,58
245,Yogyakarta siapkan lokasi servis becak listrik...,REAL,Antara News,News,59
246,Perlu upaya serius kurangi ketergantungan impo...,REAL,Antara News,News,57


💾 dataset_info.json & raw_dataset.csv disimpan


In [3]:
# Cell 3: Visualisasi Distribusi Label, Sumber, Panjang Judul
label_counts  = df['label'].value_counts()
label_pct     = df['label'].value_counts(normalize=True) * 100
source_counts = df['sumber'].value_counts()

print(f"FAKE: {label_counts['FAKE']} ({label_pct['FAKE']:.1f}%) | REAL: {label_counts['REAL']} ({label_pct['REAL']:.1f}%)")

# Pie Chart Label
fig_pie = px.pie(values=label_counts.values, names=label_counts.index,
                 title="Distribusi Label Berita",
                 color_discrete_map={'REAL':'#27AE60','FAKE':'#E74C3C'}, hole=0.4)
fig_pie.update_traces(textposition='inside', textinfo='percent+label')
fig_pie.update_layout(width=700, height=500, title_x=0.5)
fig_pie.show()

# Bar Chart Sumber
fig_bar = px.bar(x=source_counts.index, y=source_counts.values, title="Distribusi Sumber Berita",
                 labels={'x':'Sumber','y':'Jumlah'}, color=source_counts.values,
                 color_continuous_scale='viridis', text=source_counts.values)
fig_bar.update_traces(textposition='outside')
fig_bar.update_layout(width=800, height=500, title_x=0.5)
fig_bar.show()

# Box Plot Panjang Judul
fig_box = px.box(df, x='label', y='title_length', color='label',
                 title="Distribusi Panjang Judul per Label",
                 color_discrete_map={'REAL':'#27AE60','FAKE':'#E74C3C'}, points="outliers")
fig_box.update_layout(width=700, height=500, title_x=0.5)
fig_box.show()

display(pd.crosstab(df['sumber'], df['label'], margins=True))

# Save
fig_pie.write_html('flask_components/visualizations/label_distribution.html')
fig_bar.write_html('flask_components/visualizations/source_distribution.html')
fig_box.write_html('flask_components/visualizations/title_length_boxplot.html')
with open('flask_components/data/dataset_info.json', 'r', encoding='utf-8') as f:
    di = json.load(f)
di['crosstab'] = pd.crosstab(df['sumber'], df['label']).to_dict()
with open('flask_components/data/dataset_info.json', 'w', encoding='utf-8') as f:
    json.dump(di, f, indent=2, ensure_ascii=False)
print("💾 3 visualisasi distribusi disimpan")

FAKE: 98 (39.5%) | REAL: 150 (60.5%)


label,FAKE,REAL,All
sumber,,,
Antara News,0,25,25
CNN Indonesia,0,25,25
Kompas.com,0,100,100
TurnBackHoax.id,98,0,98
All,98,150,248


💾 3 visualisasi distribusi disimpan


In [4]:
# Cell 4: Analisis Kata Umum FAKE vs REAL
stop_words = {'dan','di','ke','dari','yang','untuk','dengan','pada','adalah','ini','itu',
              'akan','telah','atau','juga','tidak','dalam','oleh','dapat','sudah','seperti',
              'antara','atas','bisa','sama','satu','dua','tiga',
              'the','and','of','to','in','for','is','it','on','as','be','at','by',
              'this','have','from','they','she','her','been','than','its','his','had','has'}

def get_common_words(text, n=10):
    words = re.findall(r'\b[A-Za-z]+\b', text.lower())
    filtered = [w for w in words if w not in stop_words and len(w) > 3]
    return Counter(filtered).most_common(n)

fake_words = get_common_words(' '.join(df[df['label']=='FAKE']['judul'].astype(str)))
real_words = get_common_words(' '.join(df[df['label']=='REAL']['judul'].astype(str)))

fake_df = pd.DataFrame(fake_words, columns=['Word','Frequency'])
real_df = pd.DataFrame(real_words, columns=['Word','Frequency'])

print("Top 10 kata FAKE:"); display(fake_df)
print("Top 10 kata REAL:"); display(real_df)

fig_fake = px.bar(fake_df, x='Frequency', y='Word', orientation='h', title="Top 10 Kata - FAKE",
                  color_discrete_sequence=['#E74C3C'])
fig_fake.update_layout(width=700, height=400, title_x=0.5, yaxis={'categoryorder':'total ascending'})
fig_fake.show()

fig_real = px.bar(real_df, x='Frequency', y='Word', orientation='h', title="Top 10 Kata - REAL",
                  color_discrete_sequence=['#27AE60'])
fig_real.update_layout(width=700, height=400, title_x=0.5, yaxis={'categoryorder':'total ascending'})
fig_real.show()

# Save
fig_fake.write_html('flask_components/visualizations/top_words_fake.html')
fig_real.write_html('flask_components/visualizations/top_words_real.html')
with open('flask_components/data/word_analysis.json', 'w', encoding='utf-8') as f:
    json.dump({'top_words_fake': fake_words, 'top_words_real': real_words}, f, indent=2, ensure_ascii=False)
print("💾 word_analysis.json & visualisasi kata disimpan")

Top 10 kata FAKE:


,Word,Frequency
0,salah,81
1,israel,27
2,iran,22
3,video,21
4,indonesia,18
5,penipuan,17
6,tautan,9
7,pendaftaran,9
8,karena,9
9,perang,7


Top 10 kata REAL:


,Word,Frequency
0,kebakaran,9
1,tebet,9
2,jadi,9
3,indonesia,8
4,trump,8
5,rumah,7
6,tewas,7
7,anak,7
8,kongres,6
9,cerita,5


💾 word_analysis.json & visualisasi kata disimpan


In [5]:
# Cell 5: Data Balancing & Shuffling
counts    = df['label'].value_counts()
min_count = min(counts['FAKE'], counts['REAL'])

df_balanced = pd.concat([
    df[df['label']=='FAKE'].sample(n=min_count, random_state=42),
    df[df['label']=='REAL'].sample(n=min_count, random_state=42)
], ignore_index=True)
df_balanced = shuffle(df_balanced, random_state=42).reset_index(drop=True)

print(f"✅ Total: {len(df_balanced)} | FAKE: {min_count} (50%) | REAL: {min_count} (50%)")
print(f"🔀 Sample 10 label pertama: {df_balanced['label'].head(10).tolist()}")
display(df_balanced)

# Save
df_balanced.to_csv('balanced_dataset.csv', index=False)
df_balanced.to_csv('flask_components/data/balanced_dataset.csv', index=False)
with open('flask_components/data/balancing_info.json', 'w') as f:
    json.dump({'original_fake': int(counts['FAKE']), 'original_real': int(counts['REAL']),
               'balanced_each': int(min_count), 'total': int(len(df_balanced)),
               'random_state': 42}, f, indent=2)
print("💾 balanced_dataset.csv & balancing_info.json disimpan")

✅ Total: 196 | FAKE: 98 (50%) | REAL: 98 (50%)
🔀 Sample 10 label pertama: ['REAL', 'REAL', 'FAKE', 'FAKE', 'REAL', 'REAL', 'FAKE', 'FAKE', 'FAKE', 'REAL']


,judul,label,sumber,kategori,title_length
0,"Andai Hewan Peliharan Bisa Jadi Manusia, Ini H...",REAL,Kompas.com,News,67
1,Mengenal myBCA Keyboard: Cek Saldo hingga Tran...,REAL,Kompas.com,News,70
2,SALAH Iran Pamerkan Rudal Terbesar di Dunia,FAKE,TurnBackHoax.id,News,43
3,PENIPUAN Tautan Pendaftaran Lowongan Kerja Sup...,FAKE,TurnBackHoax.id,News,52
4,"Identitas 4 Korban Tewas Kebakaran di Tebet, S...",REAL,Kompas.com,News,64
...,...,...,...,...,...
191,4 Fitur AI yang Bisa Diakses Langsung dari Cov...,REAL,Kompas.com,News,79
192,SALAH Potret Prabowo dan Perwakilan Belanda Ba...,FAKE,TurnBackHoax.id,News,72
193,PENIPUAN Tautan Pendaftaran Bantuan Subsidi Upah,FAKE,TurnBackHoax.id,News,48
194,Menggaungkan Suara Lebih Lantang dari Negara-n...,REAL,Kompas.com,News,68


💾 balanced_dataset.csv & balancing_info.json disimpan


In [6]:
# Cell 6: Stratified Train-Test Split
X = df_balanced['judul'].values
y = df_balanced['label'].map({'FAKE': 0, 'REAL': 1}).values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

train_fake_pct = sum(y_train==0) / len(y_train) * 100
test_fake_pct  = sum(y_test==0)  / len(y_test)  * 100

print(f"✅ Train: {len(X_train)} | Test: {len(X_test)}")
print(f"Train → FAKE: {sum(y_train==0)} ({train_fake_pct:.1f}%) | REAL: {sum(y_train==1)} ({100-train_fake_pct:.1f}%)")
print(f"Test  → FAKE: {sum(y_test==0)}  ({test_fake_pct:.1f}%)  | REAL: {sum(y_test==1)}  ({100-test_fake_pct:.1f}%)")
print(f"{'✅ Stratifikasi sempurna!' if abs(train_fake_pct-test_fake_pct) < 2 else '⚠️ Perlu pengecekan'}")

# Save
pd.DataFrame({'judul': X_train, 'label': y_train}).to_csv('flask_components/data/train_data.csv', index=False)
pd.DataFrame({'judul': X_test,  'label': y_test }).to_csv('flask_components/data/test_data.csv',  index=False)
with open('flask_components/data/splitting_info.json', 'w') as f:
    json.dump({'test_size': 0.2, 'random_state': 42, 'stratify': True,
               'train_size': int(len(X_train)), 'test_size_actual': int(len(X_test)),
               'train_fake': int(sum(y_train==0)), 'train_real': int(sum(y_train==1)),
               'test_fake':  int(sum(y_test==0)),  'test_real':  int(sum(y_test==1))}, f, indent=2)
print("💾 train_data.csv, test_data.csv & splitting_info.json disimpan")

✅ Train: 156 | Test: 40
Train → FAKE: 78 (50.0%) | REAL: 78 (50.0%)
Test  → FAKE: 20  (50.0%)  | REAL: 20  (50.0%)
✅ Stratifikasi sempurna!
💾 train_data.csv, test_data.csv & splitting_info.json disimpan


In [7]:
# Cell 7: Load IndoBERT Model & Tokenizer
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

model_name = "indolem/indobert-base-uncased"
try:
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=2, output_attentions=False, output_hidden_states=False)
    print(f"✅ Model loaded: {model_name}")
except:
    model_name = "distilbert-base-uncased"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
    print(f"✅ Fallback model: {model_name}")

total_params = sum(p.numel() for p in model.parameters())
print(f"📊 Parameters: ~{total_params/1e6:.1f}M | Vocab: {tokenizer.vocab_size} | Device: {'CUDA' if torch.cuda.is_available() else 'CPU'}")

with open('flask_components/models/model_config.json', 'w') as f:
    json.dump({'model_name': model_name, 'num_labels': 2,
               'label_map': {'0':'FAKE','1':'REAL'},
               'vocab_size': int(tokenizer.vocab_size),
               'total_params': int(total_params), 'max_length': 128,
               'device': 'CUDA' if torch.cuda.is_available() else 'CPU'}, f, indent=2)
print("💾 model_config.json disimpan")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: indolem/indobert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅ Model loaded: indolem/indobert-base-uncased
📊 Parameters: ~110.6M | Vocab: 31923 | Device: CPU
💾 model_config.json disimpan


In [8]:
# Cell 8: Tokenisasi + PyTorch Dataset
def tokenize_texts(texts, max_length=128):
    return tokenizer(list(texts), truncation=True, padding=True,
                     max_length=max_length, return_tensors="pt")

train_encodings = tokenize_texts(X_train, max_length=128)
test_encodings  = tokenize_texts(X_test,  max_length=128)

class FakeNewsDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels    = labels
    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item
    def __len__(self):
        return len(self.labels)

train_dataset = FakeNewsDataset(train_encodings, y_train)
test_dataset  = FakeNewsDataset(test_encodings,  y_test)

train_mb = sum(t.nelement()*t.element_size() for t in train_encodings.values()) / 1024**2
test_mb  = sum(t.nelement()*t.element_size() for t in test_encodings.values())  / 1024**2
sample   = train_dataset[0]

print(f"✅ Tokenisasi selesai | Train: {train_mb:.2f} MB | Test: {test_mb:.2f} MB")
print(f"✅ Dataset | Train: {len(train_dataset)} | Test: {len(test_dataset)}")
print(f"   Sample label: {sample['labels'].item()} ({'FAKE' if sample['labels'].item()==0 else 'REAL'})")
gc.collect()

✅ Tokenisasi selesai | Train: 0.09 MB | Test: 0.02 MB
✅ Dataset | Train: 156 | Test: 40
   Sample label: 1 (REAL)


1226

In [10]:
# Cell 9: Training Arguments + Train Model
os.makedirs('./results', exist_ok=True)

training_args = TrainingArguments(
    output_dir='./results',
    logging_steps=5,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=10,
    eval_strategy="epoch",
    save_strategy="no",
    load_best_model_at_end=False,
    dataloader_pin_memory=False,
    remove_unused_columns=False,
    report_to="none",
    use_cpu=not torch.cuda.is_available()
)

# Hapus tokenizer= dari Trainer (tidak support di versi ini)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

print("🚀 Training dimulai...")
try:
    out = trainer.train()
    print(f"🎉 Selesai! Loss: {out.training_loss:.4f} | Steps: {out.global_step} | Waktu: {out.metrics.get('train_runtime',0):.1f}s")
    with open('flask_components/results/training_results.json', 'w') as f:
        json.dump({'final_loss': float(out.training_loss),
                   'total_steps': int(out.global_step),
                   'training_time_seconds': float(out.metrics.get('train_runtime', 0)),
                   'num_epochs': int(training_args.num_train_epochs),
                   'learning_rate': float(training_args.learning_rate),
                   'batch_size': int(training_args.per_device_train_batch_size),
                   'gradient_accumulation_steps': int(training_args.gradient_accumulation_steps),
                   'effective_batch_size': int(training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps),
                   'weight_decay': float(training_args.weight_decay),
                   'warmup_steps': int(training_args.warmup_steps),
                   'max_length': 128}, f, indent=2)
    print("💾 training_results.json disimpan")
except Exception as e:
    print(f"❌ Error: {e}")

gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

🚀 Training dimulai...


Epoch,Training Loss,Validation Loss
1,1.194167,0.639076
2,1.044924,0.398259
3,0.720059,0.342987


🎉 Selesai! Loss: 1.1022 | Steps: 60 | Waktu: 191.5s
💾 training_results.json disimpan


In [12]:
# Cell 10: Evaluasi, Metrics & Confusion Matrix
import torch
import numpy as np
from torch.utils.data import DataLoader

# Evaluasi manual tanpa trainer.evaluate()
model.eval()
all_preds = []

dataloader = DataLoader(test_dataset, batch_size=4)

with torch.no_grad():
    for batch in dataloader:
        input_ids      = batch['input_ids']
        attention_mask = batch['attention_mask']
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        preds   = torch.argmax(outputs.logits, dim=-1)
        all_preds.extend(preds.numpy())

y_pred   = np.array(all_preds)
accuracy = accuracy_score(y_test, y_pred)
report   = classification_report(y_test, y_pred, target_names=['FAKE','REAL'], output_dict=True)
cm       = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

print(f"✅ Accuracy: {accuracy:.4f} ({accuracy*100:.1f}%)")
print(classification_report(y_test, y_pred, target_names=['FAKE','REAL']))

# Visualisasi Metrics
fig_metrics = go.Figure(data=[
    go.Bar(name='Precision', x=['FAKE','REAL'],
           y=[report['FAKE']['precision'], report['REAL']['precision']],
           marker_color=['#E74C3C','#27AE60']),
    go.Bar(name='Recall', x=['FAKE','REAL'],
           y=[report['FAKE']['recall'], report['REAL']['recall']],
           marker_color=['#C0392B','#1E8449']),
    go.Bar(name='F1-Score', x=['FAKE','REAL'],
           y=[report['FAKE']['f1-score'], report['REAL']['f1-score']],
           marker_color=['#E67E22','#2980B9'])
])
fig_metrics.update_layout(barmode='group', title='Metrics per Class',
                           width=700, height=450, title_x=0.5)
fig_metrics.show()

# Confusion Matrix
fig_cm = px.imshow(cm, text_auto=True, x=['FAKE','REAL'], y=['FAKE','REAL'],
                   title="Confusion Matrix",
                   labels=dict(x="Predicted", y="Actual"),
                   color_continuous_scale='Blues')
fig_cm.update_traces(textfont_size=20)
fig_cm.update_layout(width=600, height=500, title_x=0.5)
fig_cm.show()

print(f"TN:{tn} FP:{fp} FN:{fn} TP:{tp}")
print(f"Sensitivity: {tp/(tp+fn):.4f} | Specificity: {tn/(tn+fp):.4f}")

# Save
fig_metrics.write_html('flask_components/visualizations/metrics_per_class.html')
fig_cm.write_html('flask_components/visualizations/confusion_matrix.html')

eval_data = {
    'accuracy':       float(accuracy),
    'fake_precision': float(report['FAKE']['precision']),
    'fake_recall':    float(report['FAKE']['recall']),
    'fake_f1':        float(report['FAKE']['f1-score']),
    'real_precision': float(report['REAL']['precision']),
    'real_recall':    float(report['REAL']['recall']),
    'real_f1':        float(report['REAL']['f1-score']),
    'macro_f1':       float(report['macro avg']['f1-score']),
    'weighted_f1':    float(report['weighted avg']['f1-score']),
    'confusion_matrix': {
        'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp),
        'sensitivity': float(tp/(tp+fn)),
        'specificity': float(tn/(tn+fp))
    }
}
with open('flask_components/results/evaluation_results.json', 'w') as f:
    json.dump(eval_data, f, indent=2)

print("💾 evaluation_results.json, metrics & confusion matrix disimpan")

✅ Accuracy: 0.8750 (87.5%)
              precision    recall  f1-score   support

        FAKE       0.94      0.80      0.86        20
        REAL       0.83      0.95      0.88        20

    accuracy                           0.88        40
   macro avg       0.88      0.88      0.87        40
weighted avg       0.88      0.88      0.87        40



TN:16 FP:4 FN:1 TP:19
Sensitivity: 0.9500 | Specificity: 0.8000
💾 evaluation_results.json, metrics & confusion matrix disimpan


In [13]:
# Cell 11: Save Model + Test Prediksi
def predict_news(text, model, tokenizer, max_length=128):
    model.eval()
    inputs = tokenizer(text, truncation=True, padding=True, max_length=max_length, return_tensors="pt")
    with torch.no_grad():
        probs = torch.nn.functional.softmax(model(**inputs).logits, dim=-1)
        pred  = torch.argmax(probs, dim=-1).item()
    return ("REAL" if pred==1 else "FAKE"), float(probs[0][pred].item())

# Test cases dari kode asli
test_cases = [
    ("SALAH Video Warga Israel Mengungsi di Gurun Pasir",          "FAKE"),
    ("PENIPUAN Tautan Pendaftaran CPNS Kemenhub",                  "FAKE"),
    ("HOAX: Video viral tsunami di Jakarta ternyata rekayasa!",    "FAKE"),
    ("Bank Indonesia mempertahankan suku bunga acuan 5,75 persen", "REAL"),
    ("Mayat Perempuan Ditemukan di Cisauk dalam Kondisi Tangan Terborgol", "REAL"),
]

results = []
for i, (text, expected) in enumerate(test_cases, 1):
    pred, conf = predict_news(text, model, tokenizer)
    mark = "✅" if pred==expected else "❌"
    print(f"{i}. {text[:70]}\n   → {pred} ({conf:.4f}) {mark}")
    results.append({"Text": text[:50]+"...", "Expected": expected,
                    "Predicted": pred, "Confidence": round(conf,4), "Correct": pred==expected})

display(pd.DataFrame(results))
print(f"\n🎯 Test Accuracy: {sum(r['Correct'] for r in results)/len(results):.2%}")

# Save model & tokenizer
model.save_pretrained('flask_components/models/bert_model')
tokenizer.save_pretrained('flask_components/models/bert_tokenizer')
pd.DataFrame(results).to_csv('flask_components/results/sample_test_results.csv', index=False)
print("💾 bert_model/, bert_tokenizer/ & sample_test_results.csv disimpan")

1. SALAH Video Warga Israel Mengungsi di Gurun Pasir
   → FAKE (0.8868) ✅
2. PENIPUAN Tautan Pendaftaran CPNS Kemenhub
   → FAKE (0.8128) ✅
3. HOAX: Video viral tsunami di Jakarta ternyata rekayasa!
   → FAKE (0.6613) ✅
4. Bank Indonesia mempertahankan suku bunga acuan 5,75 persen
   → REAL (0.8415) ✅
5. Mayat Perempuan Ditemukan di Cisauk dalam Kondisi Tangan Terborgol
   → REAL (0.8829) ✅


,Text,Expected,Predicted,Confidence,Correct
0,SALAH Video Warga Israel Mengungsi di Gurun Pa...,FAKE,FAKE,0.8868,True
1,PENIPUAN Tautan Pendaftaran CPNS Kemenhub...,FAKE,FAKE,0.8128,True
2,HOAX: Video viral tsunami di Jakarta ternyata ...,FAKE,FAKE,0.6613,True
3,Bank Indonesia mempertahankan suku bunga acuan...,REAL,REAL,0.8415,True
4,Mayat Perempuan Ditemukan di Cisauk dalam Kond...,REAL,REAL,0.8829,True



🎯 Test Accuracy: 100.00%


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

💾 bert_model/, bert_tokenizer/ & sample_test_results.csv disimpan


In [14]:
# Cell 12: Final Summary, Flask Config & Verifikasi
final_summary = {
    'model_info': {'model_name': model_name, 'num_labels': 2,
                   'label_map': {'0':'FAKE','1':'REAL'}, 'max_length': 128,
                   'epochs': 3, 'learning_rate': 2e-5, 'batch_size': 4},
    'dataset_info': {'total': int(len(df_balanced)), 'train': int(len(X_train)),
                     'test': int(len(X_test)), 'balance': '50-50'},
    'performance': {
        'accuracy':       float(accuracy),
        'macro_f1':       float(report['macro avg']['f1-score']),
        'fake_precision': float(report['FAKE']['precision']),
        'fake_recall':    float(report['FAKE']['recall']),
        'fake_f1':        float(report['FAKE']['f1-score']),
        'real_precision': float(report['REAL']['precision']),
        'real_recall':    float(report['REAL']['recall']),
        'real_f1':        float(report['REAL']['f1-score']),
        'sensitivity':    float(tp/(tp+fn)),
        'specificity':    float(tn/(tn+fp))
    }
}
with open('flask_components/results/final_summary.json', 'w') as f:
    json.dump(final_summary, f, indent=2)

flask_config = {
    'model_paths': {
        'bert_model':      'flask_components/models/bert_model',
        'bert_tokenizer':  'flask_components/models/bert_tokenizer',
        'model_config':    'flask_components/models/model_config.json'
    },
    'data_paths': {
        'raw_dataset':      'flask_components/data/raw_dataset.csv',
        'balanced_dataset': 'flask_components/data/balanced_dataset.csv',
        'train_data':       'flask_components/data/train_data.csv',
        'test_data':        'flask_components/data/test_data.csv',
        'dataset_info':     'flask_components/data/dataset_info.json',
        'splitting_info':   'flask_components/data/splitting_info.json',
        'word_analysis':    'flask_components/data/word_analysis.json'
    },
    'results_paths': {
        'final_summary':       'flask_components/results/final_summary.json',
        'evaluation_results':  'flask_components/results/evaluation_results.json',
        'training_results':    'flask_components/results/training_results.json',
        'sample_test_results': 'flask_components/results/sample_test_results.csv'
    },
    'visualization_paths': {
        'label_distribution':   'flask_components/visualizations/label_distribution.html',
        'source_distribution':  'flask_components/visualizations/source_distribution.html',
        'title_length_boxplot': 'flask_components/visualizations/title_length_boxplot.html',
        'top_words_fake':       'flask_components/visualizations/top_words_fake.html',
        'top_words_real':       'flask_components/visualizations/top_words_real.html',
        'confusion_matrix':     'flask_components/visualizations/confusion_matrix.html',
        'metrics_per_class':    'flask_components/visualizations/metrics_per_class.html'
    },
    'app_settings': {'title': 'BERT Fake News Detector',
                     'description': 'Deteksi berita palsu berbasis IndoBERT', 'version': '1.0.0'}
}
with open('flask_components/flask_config.json', 'w') as f:
    json.dump(flask_config, f, indent=2)

# Verifikasi semua file
print("\n🔍 VERIFIKASI FILE:")
all_paths = ([v for d in [flask_config['model_paths'], flask_config['data_paths'],
                           flask_config['results_paths'], flask_config['visualization_paths']]
              for v in d.values()])
ok = sum(1 for p in all_paths if os.path.exists(p))
for p in all_paths:
    print(f"{'✅' if os.path.exists(p) else '❌'} {p}")

print(f"\n{'='*50}")
print(f"✅ {ok}/{len(all_paths)} file tersimpan")
print(f"📊 Accuracy: {accuracy:.4f} | Macro F1: {report['macro avg']['f1-score']:.4f}")
print(f"🎉 Notebook selesai! Semua komponen siap untuk Flask.")


🔍 VERIFIKASI FILE:
✅ flask_components/models/bert_model
✅ flask_components/models/bert_tokenizer
✅ flask_components/models/model_config.json
✅ flask_components/data/raw_dataset.csv
✅ flask_components/data/balanced_dataset.csv
✅ flask_components/data/train_data.csv
✅ flask_components/data/test_data.csv
✅ flask_components/data/dataset_info.json
✅ flask_components/data/splitting_info.json
✅ flask_components/data/word_analysis.json
✅ flask_components/results/final_summary.json
✅ flask_components/results/evaluation_results.json
✅ flask_components/results/training_results.json
✅ flask_components/results/sample_test_results.csv
✅ flask_components/visualizations/label_distribution.html
✅ flask_components/visualizations/source_distribution.html
✅ flask_components/visualizations/title_length_boxplot.html
✅ flask_components/visualizations/top_words_fake.html
✅ flask_components/visualizations/top_words_real.html
✅ flask_components/visualizations/confusion_matrix.html
✅ flask_components/visualizat